In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.vae import MultiVAE

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float32

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [5]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
user_interacted_with = matrix_mf.R
user_interacted_with

tensor([[ 0.,  0., 62.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0., 84.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [44., 60.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ..., 92.,  0.,  0.]], device='mps:0')

In [6]:
user_interacted_with.shape

torch.Size([8, 923])

In [7]:
alpha = matrix_mf.compute_alpha().item()
user_interacted_with *= alpha
alpha

0.10527777671813965

In [8]:
users, items = user_interacted_with.shape

# Define and train the MultiVAE model

In [9]:
batch_size = 32

train_set = user_interacted_with
train_dataset = TensorDataset(train_set)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

In [10]:
# Define dimensions for the encoder and decoder
encoder_dims = [user_interacted_with.shape[1], 800, 400]
decoder_dims = encoder_dims[::-1]  # Symmetric decoder
model = MultiVAE(encoder_dims, decoder_dims, dropout_rate=0.5, device=device, dtype=dtype)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0)

model.train_model(
    train_loader,
    optimizer=optimizer,
    num_epochs=1000,
    max_anneal=1.0,
    anneal_steps=2000,
    log_interval=20
)

Epoch: 19, Step: 20, Loss: 4411.705078125, Anneal: 0.009999999776482582
Epoch: 39, Step: 40, Loss: 3722.799560546875, Anneal: 0.019999999552965164
Epoch: 59, Step: 60, Loss: 3698.454345703125, Anneal: 0.029999999329447746
Epoch: 79, Step: 80, Loss: 3691.42529296875, Anneal: 0.03999999910593033
Epoch: 99, Step: 100, Loss: 3690.099365234375, Anneal: 0.05000000074505806
Epoch: 119, Step: 120, Loss: 3683.949462890625, Anneal: 0.05999999865889549
Epoch: 139, Step: 140, Loss: 3679.990966796875, Anneal: 0.07000000029802322
Epoch: 159, Step: 160, Loss: 3679.43017578125, Anneal: 0.07999999821186066
Epoch: 179, Step: 180, Loss: 3680.41357421875, Anneal: 0.09000000357627869
Epoch: 199, Step: 200, Loss: 3680.61181640625, Anneal: 0.10000000149011612
Epoch: 219, Step: 220, Loss: 3681.733642578125, Anneal: 0.10999999940395355
Epoch: 239, Step: 240, Loss: 3679.70703125, Anneal: 0.11999999731779099
Epoch: 259, Step: 260, Loss: 3682.564208984375, Anneal: 0.12999999523162842
Epoch: 279, Step: 280, Loss: 

In [11]:
losses = model.losses.cpu().detach().numpy()
go.Figure(
    data=[
        go.Scatter(
            x=list(range(len(losses))),
            y=losses,
            name="Train Loss",
            mode="lines",
        ),
    ],
    layout=go.Layout(
        title="Training and Validation Loss",
        xaxis=dict(title="Epoch"),
        yaxis=dict(title="Loss"),
    ),
)

In [12]:
# retrieve users latent representations (mean and variance)
usersnames = matrix_mf.get_usernames()

model.eval()
with torch.no_grad():
    mu, logvar = model.encode(user_interacted_with)
    z = model.reparameterize(mu, logvar)
    z = z.cpu().numpy()
    mu = mu.cpu().numpy()
    std = np.exp(0.5 * logvar.cpu().numpy())
    logvar = logvar.cpu().numpy()

print(z)
print(mu)
print(logvar)

[[ 0.7959045   0.97412425 -0.15371713 ... -0.93330437  0.12344682
   0.45037666]
 [ 0.18293707  0.4592362  -0.09230876 ... -0.7157035  -0.7433417
  -1.2869257 ]
 [-0.19280237 -0.22141884 -0.26142895 ... -0.3187984  -0.6894474
  -0.4976485 ]
 ...
 [-0.43285063 -0.5090207   0.62810713 ...  0.05735752 -0.00986211
  -0.19234931]
 [ 0.14078785 -0.16540919 -0.9757839  ...  0.33930796  0.44822422
  -0.27256894]
 [ 0.16257793  0.01996675 -0.11035457 ...  0.36272255  0.02443261
  -0.20818065]]
[[ 0.7959045   0.97412425 -0.15371713 ... -0.93330437  0.12344682
   0.45037666]
 [ 0.18293707  0.4592362  -0.09230876 ... -0.7157035  -0.7433417
  -1.2869257 ]
 [-0.19280237 -0.22141884 -0.26142895 ... -0.3187984  -0.6894474
  -0.4976485 ]
 ...
 [-0.43285063 -0.5090207   0.62810713 ...  0.05735752 -0.00986211
  -0.19234931]
 [ 0.14078785 -0.16540919 -0.9757839  ...  0.33930796  0.44822422
  -0.27256894]
 [ 0.16257793  0.01996675 -0.11035457 ...  0.36272255  0.02443261
  -0.20818065]]
[[-0.03646217 -0.105

In [13]:
# Define the number of points for grid, the width of the gaussian curves and color map
if encoder_dims[-1] == 2:
        # Plot 2D latent space using and add annotations with usernames
    fig = px.scatter(x=z[:, 0], y=z[:, 1], hover_name=usersnames, color=usersnames)
    fig.update_traces(marker=dict(size=10))
    fig.update_layout(title="2D Latent Space")
    # Show the usernames next to the points
    for i, username in enumerate(usersnames):
        fig.add_annotation(
            x=z[i, 0], y=z[i, 1], text=username, showarrow=False, xshift=-20, yshift=15
        )
    fig.show()
    
    plotting.plot_multivariate_gaussian_image_with_labels(mu, std, usersnames)

In [16]:
user = "michelle"
user_idx = matrix_mf.usernames_to_ids([user])[0]
user_data = matrix_mf.get_user(user_idx).to(device).unsqueeze(0)
user_interacted_with = matrix_mf.user_interacted_with(user_idx)

recommended_items_ids = model.recommend_items(user_data, user_interacted_with, top_k=10)
recommended_items_ids

recommended_items = matrix_mf.ids_to_itemnames(recommended_items_ids)
df_recommended_items = df_matrix_mf[df_matrix_mf["id"].isin(recommended_items)]
df_recommended_items[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
21,jaslkh,Black Eyed Peas,Where Is The Love?,2003,83,0.834,0.699,0.1780,0.1110,0.000000,0.1310,0.809,94.086,-3.222,272533,2003,83
22,jaslkh,Laurent Voulzy,Le coeur grenadine,1979,48,0.721,0.292,0.0304,0.6490,0.003300,0.1790,0.540,103.349,-16.076,347520,1979,48
51,jaslkh,French 79,Lovin' Feeling,2016,73,0.000,0.609,0.0000,0.1350,0.627000,0.0778,0.000,0.000,-8.737,204933,2016,73
57,jaslkh,Black Eyed Peas,Where Is The Love?,2003,83,0.834,0.699,0.1780,0.1110,0.000000,0.1310,0.809,94.086,-3.222,272533,2003,83
60,jaslkh,Laurent Voulzy,Le coeur grenadine,1979,48,0.721,0.292,0.0304,0.6490,0.003300,0.1790,0.540,103.349,-16.076,347520,1979,48
81,jaslkh,Vendredi sur Mer,La femme à la peau bleue,2019,59,0.673,0.557,0.0446,0.5020,0.519000,0.3690,0.282,99.972,-6.645,206546,2019,59
104,jaslkh,Black Eyed Peas,Where Is The Love?,2003,83,0.834,0.699,0.1780,0.1110,0.000000,0.1310,0.809,94.086,-3.222,272533,2003,83
1510,owen,Wolf Alice,Don’t Delete The Kisses,2017,66,0.604,0.789,0.0273,0.0011,0.828000,0.3680,0.348,122.032,-6.259,275226,2017,66
1637,owen,Wolf Alice,Don’t Delete The Kisses,2017,66,0.604,0.789,0.0273,0.0011,0.828000,0.3680,0.348,122.032,-6.259,275226,2017,66
6966,brenda,Angèle,Plus de sens,2023,43,0.735,0.693,0.0390,0.3050,0.000000,0.0966,0.525,116.989,-7.719,209605,2023,43
